In [1]:
# Model Comparison and Analysis Table
import os
import re

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File locations
STGCN_FILE  = '/content/drive/MyDrive/HRC_Research/results/accuracy_logs/stgcn_hri30_70_10_20_results.txt'
CTRGCN_FILE = '/content/drive/MyDrive/HRC_Research/results/accuracy_logs/ctrgcn_hri30_70_10_20_results.txt'
OUTPUT_FILE = '/content/drive/MyDrive/HRC_Research/results/accuracy_logs/phase25_comparison_70_10_20.txt'

def parse_results(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}\nPlease check if the file exists in your Drive.")

    with open(filepath, 'r') as f:
        lines = f.readlines()

    print(f"\n--- Raw Sample from {os.path.basename(filepath)} ---")

    for line in lines[:10]:
        print(line.strip())
    print("...")

    acc_dict = {}

    pattern = re.compile(r'Class\s+(\d+).*?([\d\.]+)%')

    for line in lines:
        match = pattern.search(line)
        if match:
            class_id = int(match.group(1))
            acc = float(match.group(2))
            acc_dict[class_id] = acc

    if len(acc_dict) == 0:
        raw_snippet = "\n".join([line.strip() for line in lines[:5]])
        raise ValueError(f"Could not parse any class accuracies from {filepath}.\nRaw content snippet looks like this:\n{raw_snippet}")

    return acc_dict

try:
    print("Reading and parsing files...")
    stgcn_data = parse_results(STGCN_FILE)
    ctrgcn_data = parse_results(CTRGCN_FILE)

    if len(stgcn_data) != 30 or len(ctrgcn_data) != 30:
        print(f"\nWARNING: Expected 30 classes. Found ST-GCN: {len(stgcn_data)}, CTR-GCN: {len(ctrgcn_data)}")

    # Build comparison table and summaries
    output_lines = []
    output_lines.append("=" * 75)
    output_lines.append(f"{'Class':<6} | {'ST-GCN Acc%':<12} | {'CTR-GCN Acc%':<13} | {'Δ (CTR - ST)':<13} | {'Winner':<8}")
    output_lines.append("-" * 75)

    stgcn_wins = []
    hard_classes = []

    for i in range(30):
        st_acc = stgcn_data.get(i, 0.0)
        ctr_acc = ctrgcn_data.get(i, 0.0)
        delta = ctr_acc - st_acc

        if ctr_acc > st_acc:
            winner = "CTR-GCN"
        elif st_acc > ctr_acc:
            winner = "ST-GCN"
            stgcn_wins.append(f"{i:02d}")
        else:
            winner = "TIE"

        if st_acc < 50.0 and ctr_acc < 50.0:
            hard_classes.append(f"{i:02d}")

        output_lines.append(f"Class {i:02d} | {st_acc:>10.1f}% | {ctr_acc:>11.1f}% | {delta:>+12.1f}% | {winner}")

    output_lines.append("=" * 75)
    output_lines.append("\n--- SUMMARY STATISTICS ---")


    output_lines.append("Overall Top-1: ST-GCN = 55.27%, CTR-GCN = 67.69%, Δ = +12.42pp\n")


    sort_stgcn = sorted(stgcn_data.items(), key=lambda x: x[1])
    sort_ctrgcn = sorted(ctrgcn_data.items(), key=lambda x: x[1])

    stgcn_best_str = ", ".join([f'Class {k:02d} ({v:.1f}%)' for k,v in sort_stgcn[-5:][::-1]])
    stgcn_worst_str = ", ".join([f'Class {k:02d} ({v:.1f}%)' for k,v in sort_stgcn[:5]])

    ctrgcn_best_str = ", ".join([f'Class {k:02d} ({v:.1f}%)' for k,v in sort_ctrgcn[-5:][::-1]])
    ctrgcn_worst_str = ", ".join([f'Class {k:02d} ({v:.1f}%)' for k,v in sort_ctrgcn[:5]])

    output_lines.append(f"Best 5 classes for ST-GCN:  {stgcn_best_str}")
    output_lines.append(f"Worst 5 classes for ST-GCN: {stgcn_worst_str}\n")

    output_lines.append(f"Best 5 classes for CTR-GCN:  {ctrgcn_best_str}")
    output_lines.append(f"Worst 5 classes for CTR-GCN: {ctrgcn_worst_str}\n")

    output_lines.append(f"Classes where ST-GCN beats CTR-GCN: {', '.join(stgcn_wins) if stgcn_wins else 'None'}")
    output_lines.append(f"Classes where both models score < 50%: {', '.join(hard_classes) if hard_classes else 'None'}")

    final_text = "\n".join(output_lines)

    print("\nBuilding comparison table...\n")
    print(final_text)

    # Save to Drive
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, 'w') as f:
        f.write(final_text)

    file_size_kb = os.path.getsize(OUTPUT_FILE) / 1024
    print("\nSaving to Drive...")
    print(f"Comparison table saved to Drive.")
    print(f"File size: {file_size_kb:.2f} KB")

except Exception as e:
    print(f"\nCRITICAL ERROR: {type(e).__name__} - {str(e)}")
    print("Execution stopped.")

Mounted at /content/drive
Reading and parsing files...

--- Raw Sample from stgcn_hri30_70_10_20_results.txt ---
ST-GCN Fine-tuned on HRI30 (70/10/20 Split)
Best epoch: 47
Overall Test Accuracy: 55.27%

Per-class accuracy:
Class 00: 31.6%
Class 01: 40.0%
Class 02: 20.0%
Class 03: 70.0%
Class 04: 63.2%
...

--- Raw Sample from ctrgcn_hri30_70_10_20_results.txt ---
CTR-GCN Fine-tuned on HRI30 (70/10/20 Split)
Best epoch: 44
Overall Test Accuracy: 67.69%

Per-class accuracy:
Class 00: 63.2%
Class 01: 60.0%
Class 02: 65.0%
Class 03: 85.0%
Class 04: 68.4%
...

Building comparison table...

Class  | ST-GCN Acc%  | CTR-GCN Acc%  | Δ (CTR - ST)  | Winner  
---------------------------------------------------------------------------
Class 00 |       31.6% |        63.2% |        +31.6% | CTR-GCN
Class 01 |       40.0% |        60.0% |        +20.0% | CTR-GCN
Class 02 |       20.0% |        65.0% |        +45.0% | CTR-GCN
Class 03 |       70.0% |        85.0% |        +15.0% | CTR-GCN
Class 04 | 